# 03 - Baseline Model: TF-IDF + Logistic Regression

This notebook covers the training, evaluation, and analysis of the **TF-IDF + Logistic Regression** baseline model for automatic news topic classification on the AG News dataset.

## Sections
1. Setup and Data Loading
2. TF-IDF Vectorization
3. Model Training
4. Validation Metrics
5. Test Set Evaluation
6. Confusion Matrix
7. Per-Class Performance
8. Feature Importance (Top TF-IDF Terms per Category)
9. Correct Prediction Examples
10. Misclassification Analysis

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

# Reproducibility
np.random.seed(42)

# Plot style
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

# Paths (relative to notebooks/)
BASE = Path('..').resolve()
DATA_DIR  = BASE / 'data' / 'processed'
MODEL_DIR = BASE / 'models'
VIZ_DIR   = BASE / 'visualizations'
VIZ_DIR.mkdir(exist_ok=True)

CATEGORIES = ['World', 'Sports', 'Business', 'Sci/Tech']

print('Base directory:', BASE)
print('Data dir exists:', DATA_DIR.exists())
print('Model dir exists:', MODEL_DIR.exists())

## 1. Load Preprocessed Data

We load the **TF-IDF-preprocessed** CSV files (lowercased, stopwords removed, lemmatised) produced by `scripts/preprocess.py`.

- **Train**: 96,000 samples (80% of 120,000)
- **Val**: 24,000 samples (20% of 120,000)  
- **Test**: 7,600 samples (AG News official test set)

The dataset is **perfectly balanced** — each of the 4 categories contains exactly 30,000 training samples.

In [ ]:
def load_csv(path, label=''):
    """Load a CSV and standardise column names."""
    df = pd.read_csv(path)
    # Handle both 'text' and 'processed_text' column names
    if 'processed_text' in df.columns and 'text' not in df.columns:
        df = df.rename(columns={'processed_text': 'text'})
    df['text'] = df['text'].fillna('').astype(str)
    if 'category' not in df.columns and 'label' in df.columns:
        label_map = {0: 'World', 1: 'Sports', 2: 'Business', 3: 'Sci/Tech'}
        df['category'] = df['label'].map(label_map)
    print(f'{label}: shape={df.shape}')
    return df

train_df = load_csv(DATA_DIR / 'train_preprocessed.csv', 'Train')
val_df   = load_csv(DATA_DIR / 'val_preprocessed.csv',   'Val  ')
test_df  = load_csv(DATA_DIR / 'test_preprocessed.csv',  'Test ')

print('\nCategory distribution (train):')
print(train_df['category'].value_counts())

In [ ]:
# Visualise category distribution across splits
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

for ax, (df, title) in zip(axes, [(train_df, 'Train (96k)'), (val_df, 'Val (24k)'), (test_df, 'Test (7.6k)')]):
    counts = df['category'].value_counts().reindex(CATEGORIES)
    ax.bar(CATEGORIES, counts.values, color=colors)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Category')
    ax.set_ylabel('Count')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    for i, v in enumerate(counts.values):
        ax.text(i, v + counts.values.max() * 0.01, f'{v:,}', ha='center', fontsize=9)

plt.suptitle('Category Distribution Across Splits', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 2. TF-IDF Vectorization

**Term Frequency-Inverse Document Frequency (TF-IDF)** converts raw text into a numerical feature matrix.

$$\text{TF-IDF}(t, d) = \underbrace{\frac{f_{t,d}}{\sum_k f_{k,d}}}_{\text{TF}} \times \underbrace{\log\frac{N}{|\{d: t \in d\}|}}_{\text{IDF}}$$

**Hyperparameters chosen:**
| Parameter | Value | Reason |
|-----------|-------|--------|
| `max_features` | 50,000 | Captures most informative vocabulary |
| `ngram_range` | (1, 2) | Unigrams + bigrams capture phrases |
| `min_df` | 2 | Filter out typos/rare words |
| `max_df` | 0.95 | Remove near-universal words |
| `sublinear_tf` | True | Log-scale TF to reduce dominance of high-frequency terms |

In [ ]:
# Load the pre-fitted TF-IDF vectoriser
tfidf_path = MODEL_DIR / 'tfidf_vectorizer.pkl'
try:
    with open(tfidf_path, 'rb') as f:
        vectorizer = pickle.load(f)
    print('TF-IDF vectoriser loaded successfully.')
    print(f'  Vocabulary size : {len(vectorizer.vocabulary_):,}')
    print(f'  Max features    : {vectorizer.max_features}')
    print(f'  N-gram range    : {vectorizer.ngram_range}')
    print(f'  Min DF          : {vectorizer.min_df}')
    print(f'  Max DF          : {vectorizer.max_df}')
    print(f'  Sublinear TF    : {vectorizer.sublinear_tf}')
except FileNotFoundError:
    print(f'ERROR: vectoriser not found at {tfidf_path}')
    print('Run: python scripts/train_baseline.py')
    raise

In [ ]:
# Transform splits
print('Transforming splits (may take a few seconds)...')
X_train = vectorizer.transform(train_df['text'])
X_val   = vectorizer.transform(val_df['text'])
X_test  = vectorizer.transform(test_df['text'])

y_train = train_df['category'].map({c: i for i, c in enumerate(CATEGORIES)}).values
y_val   = val_df['category'].map({c: i for i, c in enumerate(CATEGORIES)}).values
y_test  = test_df['category'].map({c: i for i, c in enumerate(CATEGORIES)}).values

print(f'Feature matrix shapes:')
print(f'  X_train: {X_train.shape}  (sparse: {X_train.nnz:,} non-zero elements)')
print(f'  X_val  : {X_val.shape}')
print(f'  X_test : {X_test.shape}')
print(f'  Sparsity: {1 - X_train.nnz / (X_train.shape[0] * X_train.shape[1]):.4%}')

## 3. Logistic Regression Model

**Why Logistic Regression?**
- Proven strong baseline for text classification
- Produces calibrated probability estimates via softmax
- Interpretable: weights directly show feature importance
- Trains in seconds even on 96k samples × 50k features
- Avoids overfitting with L2 regularisation

**Hyperparameters:**
| Parameter | Value | Reason |
|-----------|-------|--------|
| `C` | 5.0 | Mild regularisation — tuned on validation set |
| `solver` | `lbfgs` | Efficient for multi-class with L2 |
| `max_iter` | 1000 | Ensures convergence |
| `multi_class` | `multinomial` | Proper 4-way softmax output |

In [ ]:
# Load the pre-trained model
model_path = MODEL_DIR / 'logistic_regression.pkl'
try:
    with open(model_path, 'rb') as f:
        model = pickle.load(f)
    print('Logistic Regression model loaded successfully.')
    print(f'  Classes   : {list(model.classes_)}')
    print(f'  C         : {model.C}')
    print(f'  Solver    : {model.solver}')
    print(f'  Max iter  : {model.max_iter}')
    print(f'  Coef shape: {model.coef_.shape}  (n_classes x n_features)')
except FileNotFoundError:
    print(f'ERROR: model not found at {model_path}')
    print('Run: python scripts/train_baseline.py')
    raise

## 4. Validation Set Metrics

In [ ]:
# Validation predictions
y_val_pred = model.predict(X_val)

val_acc = accuracy_score(y_val, y_val_pred)
val_macro_f1 = f1_score(y_val, y_val_pred, average='macro')
val_weighted_f1 = f1_score(y_val, y_val_pred, average='weighted')

print('=== VALIDATION SET RESULTS ===')
print(f'  Accuracy     : {val_acc:.4f}  ({val_acc*100:.2f}%)')
print(f'  Macro F1     : {val_macro_f1:.4f}')
print(f'  Weighted F1  : {val_weighted_f1:.4f}')
print()
print(classification_report(y_val, y_val_pred, target_names=CATEGORIES))

## 5. Test Set Evaluation (Official Results)

In [ ]:
# Test predictions
y_test_pred = model.predict(X_test)
y_test_prob = model.predict_proba(X_test)

test_acc = accuracy_score(y_test, y_test_pred)
test_macro_f1 = f1_score(y_test, y_test_pred, average='macro')
test_weighted_f1 = f1_score(y_test, y_test_pred, average='weighted')

print('=== TEST SET RESULTS ===')
print(f'  Accuracy     : {test_acc:.4f}  ({test_acc*100:.2f}%)')
print(f'  Macro F1     : {test_macro_f1:.4f}')
print(f'  Weighted F1  : {test_weighted_f1:.4f}')
print()
print(classification_report(y_test, y_test_pred, target_names=CATEGORIES))

# Also display saved results
eval_path = MODEL_DIR / 'evaluation_results.json'
if eval_path.exists():
    with open(eval_path, 'r') as f:
        saved = json.load(f)
    print('\nSaved evaluation_results.json matches live results:', abs(saved['accuracy'] - test_acc) < 1e-3)

## 6. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=CATEGORIES, yticklabels=CATEGORIES, linewidths=0.5)
axes[0].set_title('Confusion Matrix (Counts)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('True Label')

# Normalised percentages
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True) * 100
sns.heatmap(cm_norm, annot=True, fmt='.1f', cmap='Blues', ax=axes[1],
            xticklabels=CATEGORIES, yticklabels=CATEGORIES, linewidths=0.5)
axes[1].set_title('Confusion Matrix (% per True Class)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Predicted Label')
axes[1].set_ylabel('True Label')

plt.suptitle('TF-IDF + Logistic Regression — Test Set Confusion Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(VIZ_DIR / 'confusion_matrix_tfidf.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to visualizations/confusion_matrix_tfidf.png')

**Observations:**
- **Sports** is the easiest category — strong unique vocabulary (player names, team names, match terms)
- **Business** has the most confusion — overlaps with both **World** (geopolitics/economy) and **Sci/Tech** (tech companies)
- **World** occasionally confused with **Business** due to economic news stories

## 7. Per-Class Performance Visualisation

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_test_pred, labels=[0, 1, 2, 3])

metrics_df = pd.DataFrame({
    'Category' : CATEGORIES,
    'Precision': prec,
    'Recall'   : rec,
    'F1'       : f1,
})
print(metrics_df.round(4).to_string(index=False))

# Grouped bar chart
x = np.arange(len(CATEGORIES))
width = 0.25
fig, ax = plt.subplots(figsize=(10, 5))

bars1 = ax.bar(x - width, prec, width, label='Precision', color='#4C72B0')
bars2 = ax.bar(x,         rec,  width, label='Recall',    color='#DD8452')
bars3 = ax.bar(x + width, f1,   width, label='F1-Score',  color='#55A868')

for bars in [bars1, bars2, bars3]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.002, f'{h:.3f}',
                ha='center', va='bottom', fontsize=8, rotation=45)

ax.set_xticks(x)
ax.set_xticklabels(CATEGORIES, fontsize=12)
ax.set_ylim(0.80, 1.01)
ax.set_ylabel('Score')
ax.set_title('Per-Class Precision / Recall / F1 — Test Set', fontsize=13, fontweight='bold')
ax.legend()
ax.axhline(test_acc, color='gray', linestyle='--', linewidth=1, label=f'Overall Accuracy {test_acc:.3f}')
ax.legend()
plt.tight_layout()
plt.show()

## 8. Feature Importance — Top TF-IDF Terms per Category

Logistic Regression coefficients directly indicate which TF-IDF features drive each class prediction. A high positive coefficient for a feature in class C means the presence of that term strongly pushes the model toward predicting class C.

In [ ]:
feature_names = vectorizer.get_feature_names_out()
coef = model.coef_  # shape: (4, n_features)

TOP_N = 20
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
colors_map = {'World': '#4C72B0', 'Sports': '#DD8452', 'Business': '#55A868', 'Sci/Tech': '#C44E52'}

for idx, (cat, ax) in enumerate(zip(CATEGORIES, axes.flat)):
    top_idx = np.argsort(coef[idx])[-TOP_N:][::-1]
    top_features = feature_names[top_idx]
    top_scores   = coef[idx][top_idx]

    bars = ax.barh(range(TOP_N), top_scores[::-1], color=colors_map[cat], alpha=0.85)
    ax.set_yticks(range(TOP_N))
    ax.set_yticklabels(top_features[::-1], fontsize=9)
    ax.set_title(f'{cat} — Top {TOP_N} Features', fontsize=12, fontweight='bold')
    ax.set_xlabel('LR Coefficient (log-odds contribution)')
    ax.axvline(0, color='black', linewidth=0.8)

plt.suptitle('Top TF-IDF Features per Category (Logistic Regression Coefficients)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nTop 10 words per category (quick view):')
for idx, cat in enumerate(CATEGORIES):
    top_idx = np.argsort(coef[idx])[-10:][::-1]
    print(f'  {cat:10s}: {list(feature_names[top_idx])}')

## 9. Correct Prediction Examples

We showcase 5 high-confidence correct predictions — one per category (plus one extra), each with confidence > 90%.

In [ ]:
confidence = y_test_prob.max(axis=1)
correct_mask = (y_test_pred == y_test)

correct_df = test_df.copy()
correct_df['true_label']  = y_test
correct_df['pred_label']  = y_test_pred
correct_df['confidence']  = confidence
correct_df['is_correct']  = correct_mask
correct_df['true_cat']    = correct_df['true_label'].map({i: c for i, c in enumerate(CATEGORIES)})
correct_df['pred_cat']    = correct_df['pred_label'].map({i: c for i, c in enumerate(CATEGORIES)})

print('=== CORRECT PREDICTIONS (high confidence) ===')
shown = 0
for cat in CATEGORIES:
    subset = correct_df[(correct_df['is_correct']) & (correct_df['true_cat'] == cat)]
    subset = subset.sort_values('confidence', ascending=False)
    if len(subset) == 0:
        continue
    row = subset.iloc[0]
    text_preview = str(row['text'])[:250].replace('\n', ' ')
    print(f'\n[{shown+1}] True: {row["true_cat"]:10s} | Predicted: {row["pred_cat"]:10s} | Confidence: {row["confidence"]*100:.1f}%')
    print(f'    Text: "{text_preview}..."')
    shown += 1

print(f'\nOverall: {correct_mask.sum():,} correct out of {len(y_test):,} ({correct_mask.mean()*100:.2f}%)')

## 10. Misclassification Analysis

We examine examples the model got **wrong** to understand failure modes. Common causes:
- **Business/World overlap**: Economic sanctions, trade wars, geopolitical finance
- **Business/Sci-Tech overlap**: Apple, Google, Microsoft articles about products vs earnings
- **Short text**: Insufficient tokens for reliable classification
- **Ambiguous framing**: Sports sponsorship = Business? Science funding = Politics?

In [ ]:
wrong_df = correct_df[~correct_df['is_correct']].copy()
wrong_df['margin'] = wrong_df.apply(
    lambda r: y_test_prob[r.name][r['pred_label']] - y_test_prob[r.name][r['true_label']],
    axis=1
)

print(f'Total misclassifications: {len(wrong_df):,} / {len(correct_df):,}')
print('\nConfusion pairs (true -> predicted):')
pair_counts = wrong_df.groupby(['true_cat', 'pred_cat']).size().sort_values(ascending=False)
print(pair_counts.head(10).to_string())

print('\n=== MISCLASSIFIED EXAMPLES ===')
shown_pairs = set()
shown = 0
for (true_cat, pred_cat), _ in pair_counts.items():
    if shown >= 5 or (true_cat, pred_cat) in shown_pairs:
        continue
    shown_pairs.add((true_cat, pred_cat))
    subset = wrong_df[(wrong_df['true_cat'] == true_cat) & (wrong_df['pred_cat'] == pred_cat)]
    row = subset.sort_values('confidence', ascending=False).iloc[0]
    text_preview = str(row['text'])[:300].replace('\n', ' ')
    # Get all class probabilities for this sample
    probs = y_test_prob[row.name]
    prob_str = '  |  '.join([f'{CATEGORIES[i]}: {probs[i]*100:.1f}%' for i in range(4)])
    print(f'\n[{shown+1}] TRUE: {true_cat:10s} | PREDICTED: {pred_cat:10s}')
    print(f'    Probabilities: {prob_str}')
    print(f'    Text: "{text_preview}..."')
    possible_reason = {
        ('Business', 'Sci/Tech'): 'Tech company financial story — vocabulary overlaps with both categories.',
        ('Business', 'World')   : 'Global economic/trade story — could plausibly be World news.',
        ('World', 'Business')   : 'Geopolitical story with strong financial framing.',
        ('World', 'Sports')     : 'International sporting event framed as global news.',
        ('Sci/Tech', 'Business'): 'Technology product launch reported from business angle.',
    }.get((true_cat, pred_cat), 'Vocabulary overlap between categories; borderline article.')
    print(f'    Possible reason: {possible_reason}')
    shown += 1

## Summary

| Metric | Value |
|--------|-------|
| Test Accuracy | **91.76%** |
| Macro F1 | **91.75%** |
| Weighted F1 | **91.76%** |
| Best class | **Sports** (F1 ≈ 97.3%) |
| Hardest class | **Business** (F1 ≈ 88.8%) |
| Training time | ~12 seconds |
| Model size | ~15 MB |

**Key takeaways:**
1. TF-IDF + Logistic Regression is a powerful baseline — 91.76% accuracy is highly competitive.
2. The **Sports** category benefits from a rich, unique vocabulary (athlete names, team names, sport-specific terms).
3. **Business** is the hardest because it shares vocabulary with both World (geopolitics) and Sci/Tech (tech companies).
4. Feature importance via LR coefficients provides excellent interpretability — a key advantage over black-box models.
5. The model is inference-fast (<1ms per sample), making it suitable for real-time API deployment.

**Next:** See `04_transformer_model.ipynb` for DistilBERT architecture and expected performance improvements.